# 🕵️ Dia 51 — Forense de sistema de arquivos com The Sleuth Kit

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 51* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-51.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O `fls` do pen drive "AG CENTRO" mostrou uma linha com asterisco: `_ODULO_S.ZIP`, apagado. Rafa: *"08h35 em qual fuso?"* O FAT não sabe. Você vai ter que descobrir.

Neste notebook você vai construir um **mini Sleuth Kit** em Python e usá-lo em duas imagens:

| Ferramenta do TSK | Aqui |
|---|---|
| `fsstat` | `ler_bpb()` |
| `fls -r -p` / `fls -r -d -p` | `listar()` |
| `istat` + `icat -r` | `recuperar()` |
| `file` | `tipo_real()` |
| `fls -m` + `mactime -d -y` | `body_file()` + `mactime()` |

🐍 Só biblioteca padrão. As imagens são geradas aqui mesmo, iguais às do Dia 50.


In [ ]:
# (OPCIONAL) O TSK de verdade no Colab. Depois de rodar o notebook, você pode comparar:
# !apt-get -qq install -y sleuthkit > /dev/null
# !fls -r -p lab-p4/work/evid-004.raw
# !fls -r -m "EVID-004:" -z UTC lab-p4/work/evid-004.raw > evid-004.body && mactime -b evid-004.body -d -y -z UTC
print("Célula opcional.")


## 1. Recriar as imagens (EVID-004 e EVID-SA-004)


In [ ]:
# ------------------------------------------------------------
# fat16lab — um "pen drive" FAT16 de brinquedo, em Python puro.
# Constrói uma imagem FAT16 pequena (4 MiB) e lê essa imagem
# como The Sleuth Kit faria (fsstat, fls, istat, icat -r).
# Só biblioteca padrão. Tudo determinístico: mesma entrada,
# mesmos bytes, mesmo SHA-256.
# ------------------------------------------------------------
import struct                          # empacota números em bytes (little-endian, como o FAT)
from datetime import datetime          # datas dos arquivos

BPS = 512            # bytes por setor
SPC = 1              # setores por cluster (cluster = 512 bytes, pequeno de propósito)
RESERVADOS = 1       # só o setor de boot
NFATS = 2            # FAT e cópia da FAT, como em todo FAT16
ENTRADAS_RAIZ = 512  # a raiz do FAT16 tem tamanho fixo
TOTAL_SETORES = 8192 # 8192 x 512 = 4 MiB
SETORES_FAT = 32     # 32 x 512 / 2 bytes = 8192 entradas, cobre todos os clusters


def _data_fat(dt):
    # FAT guarda a data em 16 bits: ano desde 1980 (7), mês (4), dia (5)
    return ((dt.year - 1980) << 9) | (dt.month << 5) | dt.day


def _hora_fat(dt):
    # e a hora em 16 bits: hora (5), minuto (6), segundos/2 (5). Sem fuso!
    return (dt.hour << 11) | (dt.minute << 5) | (dt.second // 2)


def _nome83(nome):
    # "VENDORS.CSV" -> b"VENDORS CSV" (8 + 3, preenchido com espaços)
    base, _, ext = nome.upper().partition(".")
    return (base[:8].ljust(8) + ext[:3].ljust(3)).encode("ascii")


def _entrada(nome11, atributo, cluster, tamanho, criado, modificado):
    # Uma entrada de diretório tem 32 bytes. A ordem dos campos é a do FAT.
    return struct.pack(
        "<11sBBBHHHHHHHI",
        nome11, atributo, 0,
        0,                                         # décimos de segundo da criação
        _hora_fat(criado), _data_fat(criado),      # criação
        _data_fat(modificado),                     # último acesso: SÓ data
        0,                                         # cluster alto (zero no FAT16)
        _hora_fat(modificado), _data_fat(modificado),  # última escrita
        cluster, tamanho)


def construir_fat16(rotulo, id_volume, pastas, arquivos, apagar=(), criado_em=None):
    """
    rotulo: texto de até 11 letras; id_volume: inteiro de 32 bits.
    pastas: {"FINANCE": datetime}; arquivos: [("FINANCE/VENDORS.CSV", bytes, datetime_modificacao)].
    apagar: caminhos que serão "deletados" como o mdel faz (0xE5 + cadeia zerada).
    criado_em: hora de criação gravada em todos (o mtools grava a hora da cópia).
    Devolve bytearray com a imagem inteira.
    """
    img = bytearray(TOTAL_SETORES * BPS)                    # o "dispositivo", todo em zero
    inicio_fat = RESERVADOS * BPS
    inicio_raiz = inicio_fat + NFATS * SETORES_FAT * BPS
    inicio_dados = inicio_raiz + ENTRADAS_RAIZ * 32
    tam_cluster = BPS * SPC

    # --- setor de boot (BPB) ---
    boot = struct.pack("<3s8sHBHBHHBHHHII", b"\xEB\x3C\x90", b"PYLAB1.0",
                       BPS, SPC, RESERVADOS, NFATS, ENTRADAS_RAIZ, TOTAL_SETORES,
                       0xF8, SETORES_FAT, 32, 64, 0, 0)
    boot += struct.pack("<BBBI11s8s", 0x80, 0, 0x29, id_volume,
                        rotulo.upper().ljust(11)[:11].encode("ascii"), b"FAT16   ")
    img[0:len(boot)] = boot
    img[510:512] = b"\x55\xAA"                              # assinatura de setor de boot

    fat = [0] * (SETORES_FAT * BPS // 2)
    fat[0], fat[1] = 0xFFF8, 0xFFFF                         # entradas reservadas
    proximo = [2]                                           # primeiro cluster de dados é o 2

    def alocar(n):
        # aloca n clusters contíguos e encadeia na FAT
        primeiro = proximo[0]
        for i in range(n):
            c = primeiro + i
            fat[c] = 0xFFFF if i == n - 1 else c + 1
        proximo[0] += n
        return primeiro, list(range(primeiro, primeiro + n))

    def offset_cluster(c):
        return inicio_dados + (c - 2) * tam_cluster

    criado_em = criado_em or datetime(2026, 1, 1, 0, 0, 0)
    raiz = [_entrada(rotulo.upper().ljust(11)[:11].encode("ascii"), 0x08, 0, 0, criado_em, criado_em)]
    subdirs = {}                                            # nome -> (cluster, lista de entradas)
    for nome, quando in pastas.items():
        c, _ = alocar(1)
        subdirs[nome] = (c, [
            _entrada(b".          ", 0x10, c, 0, quando, quando),
            _entrada(b"..         ", 0x10, 0, 0, quando, quando)])
        raiz.append(_entrada(_nome83(nome), 0x10, c, 0, quando, quando))

    clusters_de = {}
    for item in arquivos:
        caminho, conteudo, mtime = item[:3]
        ctime = item[3] if len(item) > 3 else criado_em     # 4º campo opcional: hora de criação
        n = max(1, -(-len(conteudo) // tam_cluster))        # divisão arredondando para cima
        c, lista = alocar(n)
        clusters_de[caminho] = lista
        for i, cl in enumerate(lista):                      # grava o conteúdo cluster a cluster
            pedaco = conteudo[i * tam_cluster:(i + 1) * tam_cluster]
            o = offset_cluster(cl)
            img[o:o + len(pedaco)] = pedaco
        e = _entrada(_nome83(caminho.split("/")[-1]), 0x20, c, len(conteudo), ctime, mtime)
        if "/" in caminho:
            subdirs[caminho.split("/")[0]][1].append(e)
        else:
            raiz.append(e)

    def marcar_apagado(lista_entradas, nome11):
        for i, e in enumerate(lista_entradas):
            if e[:11] == nome11:
                lista_entradas[i] = b"\xE5" + e[1:]         # o "apagar" do FAT: 1º byte vira 0xE5

    for caminho in apagar:
        nome11 = _nome83(caminho.split("/")[-1])
        alvo = subdirs[caminho.split("/")[0]][1] if "/" in caminho else raiz
        marcar_apagado(alvo, nome11)
        for cl in clusters_de[caminho]:
            fat[cl] = 0                                     # cadeia zerada; os BYTES ficam lá

    img[inicio_raiz:inicio_raiz + 32 * len(raiz)] = b"".join(raiz)
    for nome, (c, entradas) in subdirs.items():
        o = offset_cluster(c)
        img[o:o + 32 * len(entradas)] = b"".join(entradas)
    fat_bytes = struct.pack("<%dH" % len(fat), *fat)
    for k in range(NFATS):                                  # FAT1 e FAT2 idênticas
        o = inicio_fat + k * SETORES_FAT * BPS
        img[o:o + len(fat_bytes)] = fat_bytes
    return img


In [ ]:
# ------------------------------------------------------------
# Os dois "pen drives" sintéticos do dia
# ------------------------------------------------------------
from datetime import datetime

# EVID-004 do LAB-P4: os mesmos arquivos e datas do make-evid-004.sh do original.
PNG_1x1 = (b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x06\x00\x00\x00"
           b"\x1f\x15\xc4\x89\x00\x00\x00\rIDATx\x9cc\xf8\x0f\x00\x00\x01\x01\x00\x05\x18\xd8N"
           b"\x00\x00\x00\x00IEND\xaeB`\x82")          # o PNG de 1x1 escondido atrás de .TXT
EVID_004 = dict(
    rotulo="LABUSB04", id_volume=0x4C414234,
    pastas={"FINANCE": datetime(2026, 3, 13, 9, 30, 0)},
    arquivos=[
        ("README.TXT", b"SYNTHETIC LAB MEDIA - case LAB-P4 - Example Fabrication Co. (fictional)\n",
         datetime(2026, 3, 10, 14, 5, 0)),
        ("FINANCE/VENDORS.CSV", b"# SYNTHETIC - fictional vendors for a training image\n"
         b"vendor_id,name,iban_last4,approved_by\nV-1001,Acme Test Supplies,0042,fin.mgr01\n"
         b"V-1002,Contoso Sample Metals,7719,fin.mgr01\nV-1003,Placeholder Logistics,3308,acct.lead02\n",
         datetime(2026, 3, 11, 16, 20, 0)),
        ("FINANCE/MEETING.TXT", b"SYNTHETIC - Q1 close checklist\n1. Reconcile vendor ledger\n"
         b"2. Export invoices for audit\n", datetime(2026, 3, 12, 9, 45, 0)),
        ("FINANCE/SCAN0001.TXT", PNG_1x1, datetime(2026, 3, 12, 10, 2, 0)),
    ],
    apagar=["FINANCE/VENDORS.CSV"],
    criado_em=datetime(2026, 3, 13, 9, 30, 0),       # no original, a hora em que o script rodou
)

# ➕ EVID-SA-004: o pen drive "AG CENTRO" da Serra Azul (fictício). Horas em BRASÍLIA (FAT não tem fuso).
ZIP_FALSO = (b"PK\x03\x04" + b"\x00" * 26 +
             b"OPERACAO ISCA - ARQUIVO SINTETICO. Representa o 'Modulo de Seguranca' do IscaKit. "
             b"Nao contem codigo. Nada aqui executa.\n" * 6)
EVID_SA_004 = dict(
    rotulo="AG CENTRO", id_volume=0x53413034,
    pastas={"TARIFAS": datetime(2026, 9, 1, 9, 12, 4)},
    arquivos=[
        # (caminho, conteúdo, modificado, criado). Copiar no Windows preserva o "modificado" e
        # grava como "criado" a hora da cópia: é por isso que o ZIP tem criado DEPOIS de modificado.
        ("LEIAME.TXT", b"Pen drive da agencia Centro - uso interno - Cooperativa Serra Azul (ficticia)\n",
         datetime(2026, 9, 1, 9, 12, 0), datetime(2026, 9, 1, 9, 12, 0)),
        ("TARIFAS/TAB2026.CSV", b"servico,valor\nTED,0.00\nDOC,0.00\nSaque extra,2.50\n",
         datetime(2026, 9, 30, 16, 40, 12), datetime(2026, 9, 30, 16, 40, 12)),
        ("TARIFAS/AVISO.TXT", b"Atualizar a tabela no caixa 2 toda segunda-feira.\n",
         datetime(2026, 10, 2, 11, 5, 30), datetime(2026, 10, 2, 11, 5, 30)),
        ("MODULO_S.ZIP", ZIP_FALSO, datetime(2026, 10, 5, 8, 23, 14), datetime(2026, 10, 5, 8, 35, 10)),
    ],
    apagar=["MODULO_S.ZIP"],
    criado_em=datetime(2026, 10, 5, 8, 35, 10),
)
print("Receitas prontas: EVID_004 (LAB-P4) e EVID_SA_004 (pen drive AG CENTRO).")


In [ ]:
import hashlib
from pathlib import Path

IMAGENS = {}
for nome, receita, pasta in [("evid-004.raw", EVID_004, "lab-p4/work"),
                             ("evid-sa-004.raw", EVID_SA_004, "caso-isca/work")]:
    Path(pasta).mkdir(parents=True, exist_ok=True)
    p = Path(pasta) / nome
    p.write_bytes(construir_fat16(**receita))
    IMAGENS[nome] = p.read_bytes()             # trabalhamos sempre sobre BYTES lidos, nunca escrevemos na imagem
    print(f"{hashlib.sha256(IMAGENS[nome]).hexdigest()}  {p}")


## 2. `mmls` e `fsstat`

O nosso volume não tem tabela de partições (como o EVID-004 do original): o setor 0 já é o setor de boot do FAT16. Um MBR teria a assinatura `55 AA` também, mas a tabela de partições fica nos bytes 446-509 e o setor 0 não começaria com o salto `EB 3C 90`.


In [ ]:
# ------------------------------------------------------------
# Um "mini Sleuth Kit" para FAT16, em Python puro.
# fsstat -> ler_bpb | fls -r -p -> listar | istat -> entrada | icat -r -> recuperar
# Só LÊ a imagem (bytes). Nunca escreve nela.
# ------------------------------------------------------------
import struct
from datetime import datetime


def ler_bpb(img):
    # O setor de boot diz a geometria do volume: é o que o fsstat imprime.
    (bps, spc, res, nfats, nraiz, total, _mid, spf) = struct.unpack_from("<HBHBHHBH", img, 11)
    id_vol, = struct.unpack_from("<I", img, 39)
    rotulo = img[43:54].decode("ascii").strip()
    tipo = img[54:62].decode("ascii").strip()
    inicio_fat = res * bps
    inicio_raiz = inicio_fat + nfats * spf * bps
    inicio_dados = inicio_raiz + nraiz * 32
    return dict(bps=bps, spc=spc, nfats=nfats, nraiz=nraiz, total=total, spf=spf,
                id_volume=id_vol, rotulo=rotulo, tipo=tipo, inicio_fat=inicio_fat,
                inicio_raiz=inicio_raiz, inicio_dados=inicio_dados,
                tam_cluster=bps * spc)


def _dt(data, hora=0):
    # Decodifica data/hora do FAT. Sem fuso: quem decide o fuso é VOCÊ (o -z do fls).
    if data == 0:
        return None
    ano, mes, dia = 1980 + (data >> 9), (data >> 5) & 15, data & 31
    h, m, s = hora >> 11, (hora >> 5) & 63, (hora & 31) * 2
    return datetime(ano, mes, dia, h, m, s)


def _entradas(img, bpb, offset, quantidade):
    # Lê entradas de 32 bytes a partir de um offset; para no primeiro byte 0x00 (fim).
    for i in range(quantidade):
        o = offset + i * 32
        e = img[o:o + 32]
        if e[0] == 0x00:
            break
        nome = e[0:8].decode("latin-1").rstrip()
        ext = e[8:11].decode("latin-1").rstrip()
        atributo = e[11]
        (_, _, hc, dc, da, _, hm, dm, cluster, tamanho) = struct.unpack_from("<BBHHHHHHHI", e, 12)
        apagado = e[0] == 0xE5
        if apagado:
            nome = "_" + nome[1:]          # o TSK mostra o 1º caractere perdido como "_"
        yield dict(nome=nome + ("." + ext if ext else ""), atributo=atributo,
                   apagado=apagado, cluster=cluster, tamanho=tamanho,
                   criado=_dt(dc, hc), acesso=_dt(da), modificado=_dt(dm, hm), offset=o)


def listar(img):
    """Equivale a fls -r -p: devolve todas as entradas, inclusive apagadas."""
    bpb = ler_bpb(img)
    saida = []

    def visitar(offset, quantidade, prefixo):
        for e in _entradas(img, bpb, offset, quantidade):
            if e["atributo"] & 0x08 or e["nome"] in (".", ".."):
                continue                    # rótulo do volume e atalhos . e ..
            e["caminho"] = prefixo + e["nome"]
            # "inode" no estilo TSK para FAT: posição da entrada em blocos de 32 bytes
            e["inode"] = e["offset"] // 32
            saida.append(e)
            if e["atributo"] & 0x10 and not e["apagado"]:
                o = bpb["inicio_dados"] + (e["cluster"] - 2) * bpb["tam_cluster"]
                visitar(o, bpb["tam_cluster"] // 32, e["caminho"] + "/")

    visitar(bpb["inicio_raiz"], bpb["nraiz"], "")
    return saida


def recuperar(img, entrada):
    """Equivale a icat -r: lê 'tamanho' bytes a partir do 1º cluster, supondo contiguidade."""
    bpb = ler_bpb(img)
    o = bpb["inicio_dados"] + (entrada["cluster"] - 2) * bpb["tam_cluster"]
    return bytes(img[o:o + entrada["tamanho"]])


In [ ]:
img = IMAGENS["evid-004.raw"]
print("Primeiros 3 bytes:", img[:3].hex(" "), "-> salto de boot do FAT: sem tabela de partições, offset 0")

bpb = ler_bpb(img)
print(f"""
FILE SYSTEM INFORMATION (à moda do fsstat)
--------------------------------------------
File System Type: {bpb['tipo']}
Volume ID: {hex(bpb['id_volume'])}
Volume Label (Boot Sector): {bpb['rotulo']}
Sector Size: {bpb['bps']}
Cluster Size: {bpb['tam_cluster']}
Total Sector Range: 0 - {bpb['total'] - 1}
FAT 0: {bpb['inicio_fat'] // bpb['bps']} - {bpb['inicio_fat'] // bpb['bps'] + bpb['spf'] - 1}
Root Directory: setor {bpb['inicio_raiz'] // bpb['bps']}
Data Area: a partir do setor {bpb['inicio_dados'] // bpb['bps']}""")


## 3. `fls -r -p`: nomes vivos e apagados

O `*` marca o apagado. Repare no nome: `_ENDORS.CSV`. A primeira letra foi sobrescrita pelo `0xE5`; o TSK mostra o que sobrou. (Em FAT com nomes longos, o TSK costuma recuperar o nome completo pela entrada LFN; o nosso construtor só grava nomes 8.3.)


In [ ]:
def fls(img, so_apagados=False):
    for e in listar(img):
        if so_apagados and not e["apagado"]:
            continue
        tipo = "d/d" if e["atributo"] & 0x10 else "r/r"
        print(f"{tipo} {'* ' if e['apagado'] else ''}{e['inode']}:\t{e['caminho']}")

print("fls -r -p"); fls(img)
print("\nfls -r -d -p"); fls(img, so_apagados=True)


## 4. `istat` e `icat -r`

Primeiro os metadados, depois o conteúdo. A recuperação lê `tamanho` bytes a partir do primeiro *cluster*, **supondo que o arquivo era contíguo** (a cadeia na FAT foi zerada).


In [ ]:
def istat(img, e):
    bpb = ler_bpb(img)
    n_clusters = -(-e["tamanho"] // bpb["tam_cluster"])
    primeiro_setor = (bpb["inicio_dados"] + (e["cluster"] - 2) * bpb["tam_cluster"]) // bpb["bps"]
    print(f"""Directory Entry: {e['inode']}
{'Not Allocated' if e['apagado'] else 'Allocated'}
Name: {e['nome']}
Size: {e['tamanho']}
Written:  {e['modificado']}
Accessed: {e['acesso'].date() if e['acesso'] else '-'}  (FAT: só a data)
Created:  {e['criado']}
Sectors: {', '.join(str(primeiro_setor + k) for k in range(n_clusters * bpb['spc']))}""")

vendors = next(e for e in listar(img) if e["apagado"])
istat(img, vendors)
recuperado = recuperar(img, vendors)
Path("lab-p4/work/recovered_VENDORS.CSV").write_bytes(recuperado)
print("\nSHA-256 do recuperado:", hashlib.sha256(recuperado).hexdigest(), f"(inode {vendors['inode']})")
print(recuperado.decode())

# E se o arquivo estivesse fragmentado? A FAT diria. Mas a cadeia foi zerada:
bpb = ler_bpb(img)
entrada_fat = int.from_bytes(img[bpb["inicio_fat"] + 2 * vendors["cluster"]: bpb["inicio_fat"] + 2 * vendors["cluster"] + 2], "little")
print(f"Entrada da FAT para o cluster {vendors['cluster']}: {entrada_fat} (0 = livre: a cadeia sumiu, a recuperação supõe contiguidade)")


## 5. Assinatura × extensão (o `file` em miniatura)


In [ ]:
ASSINATURAS = {
    b"\x89PNG\r\n\x1a\n": "PNG image data", b"\xff\xd8\xff": "JPEG image data", b"%PDF-": "PDF document",
    b"PK\x03\x04": "Zip archive data", b"MZ": "PE executable (MS Windows)",
}

def tipo_real(conteudo):
    """Decide pelo começo dos bytes; se nada bater e for texto imprimível, diz ASCII text."""
    for sig, nome in ASSINATURAS.items():
        if conteudo.startswith(sig):
            return nome
    try:
        conteudo.decode("ascii")
        return "ASCII text"
    except UnicodeDecodeError:
        return "data"

def exportar_todos(img):
    """O nosso tsk_recover -e: vivos e apagados."""
    return [(e, recuperar(img, e)) for e in listar(img) if not e["atributo"] & 0x10]

for e, dados in exportar_todos(img):
    ext = e["nome"].rsplit(".", 1)[-1]
    t = tipo_real(dados)
    alerta = "  ⚠️ extensão não bate com a assinatura" if (ext == "TXT" and t != "ASCII text") else ""
    print(f"{e['caminho']:24} {t}{alerta}")


> 📝 Isto é **observação**, não conclusão: "SCAN0001.TXT tem assinatura PNG". Não escreva "o usuário escondeu uma imagem".

## 6. *Body file* e `mactime`

O `fls -m` grava uma linha por arquivo no formato *body* do TSK:
`MD5|nome|inode|modo|UID|GID|tamanho|atime|mtime|ctime|crtime` (tempos em *epoch*, segundos desde 1970 em UTC).

Aqui está o ponto do dia: para transformar uma hora FAT (local, sem fuso) em *epoch*, alguém tem que **escolher um fuso**. É o `-z` do `fls`.


In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo           # fusos da IANA, na biblioteca padrão desde o Python 3.9

def para_epoch(dt_local, fuso):
    """Hora FAT (ingênua) + fuso escolhido -> segundos UTC. É AQUI que o erro de fuso entra, em silêncio."""
    if dt_local is None:
        return 0
    return int(dt_local.replace(tzinfo=ZoneInfo(fuso)).timestamp())

def body_file(img, prefixo, fuso):
    linhas = []
    for e in listar(img):
        nome = f"{prefixo}/{e['caminho']}" + (" (deleted)" if e["apagado"] else "")
        modo = ("d/drwxrwxrwx" if e["atributo"] & 0x10 else "r/rrwxrwxrwx")
        # FAT não tem "changed" (ctime): o TSK grava 0
        linhas.append(f"0|{nome}|{e['inode']}|{modo}|0|0|{e['tamanho']}|"
                      f"{para_epoch(e['acesso'], fuso)}|{para_epoch(e['modificado'], fuso)}|0|"
                      f"{para_epoch(e['criado'], fuso)}")
    return linhas

def mactime(body, fuso_exibicao="UTC"):
    """Agrupa por instante: as letras m, a, c, b dizem quais tempos caíram naquele segundo."""
    eventos = {}
    for linha in body:
        _, nome, inode, modo, uid, gid, tam, a, m, c, b = linha.split("|")
        for letra, t in zip("macb", (m, a, c, b)):
            t = int(t)
            if t == 0:
                continue
            chave = (t, nome)
            ev = eventos.setdefault(chave, dict(tam=tam, modo=modo, inode=inode, flags=set()))
            ev["flags"].add(letra)
    saida = ["Date,Size,Type,Mode,UID,GID,Meta,File Name"]
    for (t, nome), ev in sorted(eventos.items()):
        quando = datetime.fromtimestamp(t, ZoneInfo(fuso_exibicao)).strftime("%Y-%m-%d %H:%M:%S")
        tipo = "".join(l if l in ev["flags"] else "." for l in "macb")
        saida.append(f"{quando},{ev['tam']},{tipo},{ev['modo']},0,0,{ev['inode']},{nome}")
    return saida

body = body_file(img, "EVID-004:", "UTC")
print("\n".join(body[:3]), "\n...\n")
linha_do_tempo = mactime(body, "UTC")
print("\n".join(linha_do_tempo))


### Passo 7 do original: `observation` e `evidence_ref`

Só linhas de 2026, mais duas colunas suas. As observações sugeridas abaixo são um começo: reescreva com as suas palavras.


In [ ]:
import csv, io

def anotar(linhas, observacoes):
    leitor = csv.reader(linhas)
    cab = next(leitor) + ["observation", "evidence_ref"]
    saida = [cab]
    for row in leitor:
        if not row[0].startswith("2026"):
            continue
        nome = row[7]
        caminho = nome.split(":", 1)[1].replace(" (deleted)", "")   # "EVID-004:/FINANCE/X" -> "/FINANCE/X"
        obs = observacoes.get(caminho, "")                            # chave = caminho exato
        saida.append(row + [obs + (" [APAGADO]" if "(deleted)" in nome else ""), f"inode {row[6]}"])
    return saida

OBS_004 = {
    "/FINANCE": "pasta FINANCE",
    "/FINANCE/_ENDORS.CSV": "tabela de fornecedores; apagada depois (recuperada, ver hash)",
    "/FINANCE/MEETING.TXT": "checklist de fechamento", "/FINANCE/SCAN0001.TXT": "nome .TXT, assinatura PNG",
    "/README.TXT": "README do volume",
}
tabela = anotar(linha_do_tempo, OBS_004)
buf = io.StringIO(); csv.writer(buf).writerows(tabela)
Path("lab-p4/work/evid-004-timeline.csv").write_text(buf.getvalue(), encoding="utf-8")
for r in tabela:
    print(" | ".join(r))


## 7. ➕ O ZIP da Serra Azul e o fuso

Agora o pen drive "AG CENTRO". O proxy da cooperativa (em UTC, Dia 49) mostra o clique no `lnk.example/sa7` às **11:19:38 UTC** e o download do "módulo" às **11:23:12 UTC**. Escolha o fuso do `fls -z` e veja onde o ZIP cai.


In [ ]:
#@title Fuso para o fls -z { display-mode: "form" }
fuso_do_pen_drive = "UTC"  #@param ["UTC", "America/Sao_Paulo"]

img_sa = IMAGENS["evid-sa-004.raw"]
CLIQUE_UTC = datetime(2026, 10, 5, 11, 19, 38, tzinfo=timezone.utc)   # proxy-sa.log, EVID-SA-001

def onde_cai_o_zip(fuso):
    body = body_file(img_sa, "EVID-SA-004:", fuso)
    tl = mactime(body, "UTC")
    for linha in tl:
        if "ODULO" in linha and "b" in linha.split(",")[2]:     # o evento de criação (b = born)
            quando = datetime.strptime(linha.split(",")[0], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
            return linha, quando
    return None, None

for fuso in ("UTC", "America/Sao_Paulo"):
    linha, quando = onde_cai_o_zip(fuso)
    rel = "DEPOIS" if quando > CLIQUE_UTC else "ANTES"
    marca = "👉" if fuso == fuso_do_pen_drive else "  "
    print(f"{marca} fls -z {fuso:18} -> criação do ZIP em {quando:%H:%M:%S} UTC, {rel} do clique (11:19:38 UTC)")

print("\nLinha do tempo do EVID-SA-004 com o fuso escolhido:")
tl_sa = mactime(body_file(img_sa, "EVID-SA-004:", fuso_do_pen_drive), "UTC")
for l in tl_sa:
    if l.startswith(("Date", "2026-10")):
        print(l)


> 🧭 Com `-z UTC`, o ZIP aparece às 08:35 UTC, **três horas antes** do clique: um "fato" impossível que alguém poderia ler como "o arquivo já estava lá antes do golpe". Com `America/Sao_Paulo`, ele cai às 11:35 UTC, doze minutos depois do download no proxy. A fonte independente (o proxy, em UTC) decide. Escreva isso na nota de fuso.

E o "modificado antes de criado"? Veja o `istat`:


In [ ]:
zip_e = next(e for e in listar(img_sa) if e["apagado"])
istat(img_sa, zip_e)
dados_zip = recuperar(img_sa, zip_e)
Path("caso-isca/work/recuperado_MODULO_S.ZIP").write_bytes(dados_zip)
print("\nSHA-256:", hashlib.sha256(dados_zip).hexdigest())
print("Assinatura:", tipo_real(dados_zip))
print("Texto dentro (é sintético):", dados_zip[30:120].decode(errors="replace"))
print("\nCriado > Modificado?", zip_e["criado"] > zip_e["modificado"],
      "-> padrão de CÓPIA no Windows: a cópia guarda o 'modificado' do original e grava 'criado' na hora da cópia.")


## 8. 🎮 Quiz: camadas do TSK e o que cada uma prova


In [ ]:
MODO_JOGO = True

PERGUNTAS = [
    ("Qual ferramenta lista nomes, inclusive apagados?", ["istat", "fls", "mmls"], 2),
    ("Qual tenta recuperar o CONTEÚDO de um arquivo apagado?", ["icat -r", "fls -d", "fsstat"], 1),
    ("No FAT, apagar um arquivo...", ["zera os clusters", "troca o 1º byte do nome por 0xE5 e zera a cadeia na FAT",
                                      "move para a lixeira do volume"], 2),
    ("Por que um apagado fragmentado no FAT não volta limpo?",
     ["porque o FAT criptografa", "porque a cadeia sumiu e a ferramenta supõe contiguidade", "porque o TSK não lê FAT"], 2),
    ("Horários do FAT estão em...", ["UTC", "hora local, sem fuso gravado", "hora do servidor NTP"], 2),
    ("SCAN0001.TXT com assinatura PNG prova que...",
     ["o usuário escondeu uma imagem", "nada sozinho: é uma observação", "é malware"], 2),
    ("➕ O que decidiu o fuso do pen drive AG CENTRO?",
     ["o palpite do analista", "uma fonte independente em UTC (o proxy)", "o nome do arquivo"], 2),
]
acertos = 0
for i, (p, ops, certa) in enumerate(PERGUNTAS, 1):
    print(f"\n[{i}] {p}")
    for k, o in enumerate(ops, 1):
        print(f"   {k}. {o}")
    if MODO_JOGO:
        ok = input("   Resposta: ").strip() == str(certa)
        acertos += ok
        print("   ✅" if ok else f"   ❌ certa: {certa}")
    else:
        print(f"   certa: {certa}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(PERGUNTAS)}")


In [ ]:
# Baixar os artefatos no Colab
try:
    from google.colab import files
    for f in ("lab-p4/work/evid-004-timeline.csv", "lab-p4/work/recovered_VENDORS.CSV"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: arquivos em lab-p4/work/ e caso-isca/work/)")


## ✅ Antes de fechar o Dia 51

- [ ] linha do tempo com a criação de `FINANCE`, as quatro gravações e o `VENDORS.CSV (deleted)`
- [ ] hash do `VENDORS.CSV` recuperado ao lado do *inode*
- [ ] nota de fuso: FAT = hora local; como achar o *offset* real
- [ ] ➕ `MODULO_S.ZIP` na linha do tempo da Serra Azul com `America/Sao_Paulo`, justificado pelo proxy
- [ ] ➕ frase do achado: "gravado no pen drive", nunca "a caixa copiou"

**Amanhã (Dia 52):** NTFS. Dois conjuntos de horários por arquivo, e um deles qualquer programa consegue falsificar.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
